# Predicting Bird Geolocation Region from Latitude and Longitude

This notebook maps bird geolocations (latitude and longitude) to North American regions: US states, Canadian provinces, and other North American countries. It trains a classifier to predict the region and visualizes the results using folium.

## 1. Import Required Libraries
We use pandas, scikit-learn, folium, and geopandas for geospatial mapping and region assignment across North America.

In [6]:
import pandas as pd
import numpy as np
import folium
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

## 2. Load and Inspect Data
Load the NABBP dataset including 'lat_dd' and 'lon_dd'. Birds may be located anywhere in North America (US, Canada, Mexico, Caribbean, Central America).

In [8]:

import os
from birds.settings import load_settings
settings = load_settings()
DATA_PATH = os.path.join(settings.nabbp_data_path, "NABBP_2025_grp_01.csv.gz")
df = pd.read_csv(DATA_PATH, compression='gzip', low_memory=True, engine='pyarrow')
df[['lat_dd', 'lon_dd']].head()

,lat_dd,lon_dd
0,58.25000,-116.41667
1,52.58333,-110.41667
2,53.41667,-110.91667
3,56.25000,-117.25000
4,51.25000,-111.75000


## Visualize Raw Geolocations with a Folium Map
Before preprocessing, let's visualize a sample of bird datapoints from the entire North American region on a map using folium.

In [ ]:
# Plot a random sample of 5000 raw datapoints on a folium map and save to HTML
n_points = 5000
sample_df = df.sample(n=n_points, random_state=42) if len(df) > n_points else df
raw_map = folium.Map(location=[40, -100], zoom_start=3)  # Center of North America
for _, row in sample_df.iterrows():
    if pd.notnull(row['lat_dd']) and pd.notnull(row['lon_dd']):
        folium.CircleMarker(
            location=[row['lat_dd'], row['lon_dd']],
            radius=2,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.5
        ).add_to(raw_map)
raw_map.save('north_america_raw_geolocations_map.html')
print('Map saved to north_america_raw_geolocations_map.html')

Map saved to raw_geolocations_map.html


## 3. Preprocess Geolocation Data
Filter out rows with missing or invalid latitude/longitude values and handle outliers. Use North American bounds for filtering.

In [ ]:
# Filter for valid North American latitude and longitude, and remove null island (lat=0, lon=0)
na_lat_bounds = (7, 84)
na_lon_bounds = (-168, -52)
df = df.dropna(subset=['lat_dd', 'lon_dd'])
df = df[(df['lat_dd'] >= na_lat_bounds[0]) & (df['lat_dd'] <= na_lat_bounds[1])]
df = df[(df['lon_dd'] >= na_lon_bounds[0]) & (df['lon_dd'] <= na_lon_bounds[1])]
df = df[~((df['lat_dd'] == 0) & (df['lon_dd'] == 0))]
df[['lat_dd', 'lon_dd']].describe()

## 4. Map Coordinates to North American Regions
Assign each (lat_dd, lon_dd) to a US state, Canadian province, or other North American country using a shapefile and geopandas.

In [19]:
# Download North America KML if not present
kml_url = "https://www.inaturalist.org/places/geometry/north-america.kml"
kml_path = "north_america.kml"
if not os.path.exists(kml_path):
    urllib.request.urlretrieve(kml_url, kml_path)

# Load KML and map coordinates to regions
regions_gdf = gpd.read_file(kml_path, driver='KML')
points_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon_dd'], df['lat_dd']), crs=regions_gdf.crs)
joined = gpd.sjoin(points_gdf, regions_gdf, how="left", predicate="within")
df['region'] = joined['Name']  # KML usually uses 'Name' for region name
df[['lat_dd', 'lon_dd', 'region']].head()


,lat_dd,lon_dd,region
0,58.25000,-116.41667,North America Border
1,52.58333,-110.41667,North America Border
2,53.41667,-110.91667,North America Border
3,56.25000,-117.25000,North America Border
4,51.25000,-111.75000,North America Border


## 5. Train a Classification Model to Predict Region
Train a RandomForest classifier to predict the region (US state, Canadian province, or other North American country) from latitude and longitude.

In [ ]:
# Prepare data for classification
X = df[['lat_dd', 'lon_dd']]
y = df['region']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train RandomForest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

## 6. Evaluate Model Performance
Assess accuracy and display a confusion matrix for the region predictions.

In [ ]:
# Evaluate accuracy and show confusion matrix
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f}")
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
plt.figure(figsize=(16,10))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Region')
plt.ylabel('Actual Region')
plt.xticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y), rotation=90)
plt.yticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y))
plt.colorbar()
plt.show()

## 7. Visualize Predictions on a Folium Map
Plot a sample of predicted vs. actual regions on an interactive folium map for North America.

In [ ]:
# Visualize a sample of predictions on a folium map
sample = X_test.copy()
sample['actual_region'] = y_test.values
sample['predicted_region'] = y_pred
sample = sample.sample(n=200, random_state=42)

m = folium.Map(location=[40, -100], zoom_start=3)  # Center of North America
for _, row in sample.iterrows():
    color = 'green' if row['actual_region'] == row['predicted_region'] else 'red'
    folium.CircleMarker(
        location=[row['lat_dd'], row['lon_dd']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"Actual: {row['actual_region']}, Predicted: {row['predicted_region']}"
    ).add_to(m)
m.save('north_america_predicted_vs_actual_map.html')
print('Map saved to north_america_predicted_vs_actual_map.html')